<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/etl_siconfi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook para extração de bases - Dimensão Fiscal / SICONFI e SIDRA(IBGE)
>Trata-se de notebook para extração de dados porovenientes das eleições do TSE dos anos de 2018 e 2022.

  
  **Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026

Observação: Agradecimentos aos idalizadores dos Pacotes aqui utilizados:


*   siconfir
*   sidrar



## Etapa Inicial: Instalação de Pacotes

In [1]:
if (!require(pacman)) install.packages("pacman")
pacman::p_load("sidrar","siconfi","httr", "jsonlite", "stringr", "dplyr", "tidyr", "knitr", "devtools", "ggplot2")


devtools::install_github("tchiluanda/rsiconfi")

library(rlang)
library(sidrar)
library(rsiconfi)
library(dplyr)
library(purrr)
library(tidyr)

Loading required package: pacman

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘pacman’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘rjson’



sidrar installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message:
“package ‘siconfi’ is not available for this version of R

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”
Warning message:
“'BiocManager' not available.  Could not check Bioconductor.

Please use `install.packages('BiocManager')` and then retry.”
Warning message in p_install(package, character.only = TRUE, ...):
“”
Warning message in library(package, lib.loc = lib.l

vctrs (0.7.0 -> 0.7.1) [CRAN]
dplyr (1.1.4 -> 1.2.0) [CRAN]


Installing 2 packages: vctrs, dplyr

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpoCgt6t/remotes93c26fb39d3/tchiluanda-rsiconfi-cb10580/DESCRIPTION’ ... OK
* preparing ‘rsiconfi’:
* checking DESCRIPTION meta-information ... OK
Warning in person1(given = given[[i]], family = family[[i]], middle = middle[[i]],  :
  Invalid ORCID iD: ‘YOUR-ORCID-ID’.
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building ‘rsiconfi_0.0.0.9000.tar.gz’



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘rlang’


The following objects are masked from ‘package:jsonlite’:

    flatten, unbox



Attaching package: ‘purrr’


The following objects are masked from ‘package:rlang’:

    flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice


The following object is masked from ‘package:jsonlite’:

    flatten




### 1ª Fase - Extrair 7 variaveis que melhor diganosticam se um estado está se endividando.
1. **Dívida Consolidada Líquida** (*dcl*) — valor absoluto da dívida menos disponibilidades (recursos em curto prazo disponíveis em caixa/estoque do governo). Proveniente do Anexo 2 do RGF.
1. **População do Estado no Ano** ( *pop* ) - trata-se da população que foi informada pelo Ente no RGF daquele ano.
1. **PIB de Preços Correntes** ( *pib_prec_corr* ) - trata-se de PIB de todas as receitas e riquezas apuradas pelo ente naquele ano. Dado extraído do SIDRA/IBGE, por isso o dado do ano 2024 está ainda indisponível até o presente momento da pesquisa. Na falta do dado do ano de 2024, atribuído valor 0.
1. **Receita Corrente Líquida** ( rcl ) - trata-se de valor absoluto da Receita Corrente Líquida do ente.
1. **Receita Corrente Líquida Ajustada** ( rcla ) - trata-se do valor de  Receita Corrente Líquida reduzido do montante de Transferências Obrigatórias da União Relativas às Emendas Individuais (art. 166-A, §1º, da CF).
1. **Despesa Total com Pessoal** (dtp) - Somatório de gastos com pessoal.
1. **Despesa Total com Pessoal como % da RCLA** (dtp_perc_rcla) - Razão entre a despesa total com pessoal (dtp) em relação ao valor da Receita Corrente Líquida Ajustada (rcla) em percentual. Proxy de rigidez fiscal embasado em norma (art. 20 e 22, da LRF), que estabelece governadores não podem comprometer mais de 49%.

In [2]:
anos <- 2019:2024

estados <- tibble(
  uf = c("RO","AC","AM","RR","PA","AP","TO",
         "MA","PI","CE","RN","PB","PE","AL","SE","BA",
         "MG","ES","RJ","SP","PR","SC","RS",
         "MS","MT","GO","DF"),
  cod_ibge = c("11","12","13","14","15","16","17",
               "21","22","23","24","25","26","27","28","29",
               "31","32","33","35","41","42","43",
               "50","51","52","53")
)


# ── Função por estado × ano ───────────────────────────────────────────────────

extrair_rgf <- function(uf, cod_ibge, ano) {

  message(sprintf(">>> %s | %d", uf, ano))

  anexo1 <- tryCatch(
    get_rgf(
      year        = ano,
      periodicity = "Q",
      period      = 3,
      report_tp   = 1,
      annex       = "01",
      entity      = cod_ibge,
      co_power    = "E"
    ),
    error = function(e) {
      warning(sprintf("Extração do Anexo 01! Erro em %s/%d: %s", uf, ano, e$message))
      return(NULL)
    }
  )


  anexo2 <- tryCatch(
    get_rgf(
      year        = ano,
      periodicity = "Q",
      period      = 3,
      report_tp   = 1,
      annex       = "02",
      entity      = cod_ibge,
      co_power    = "E"
    ),
    error = function(e) {
      warning(sprintf("Extração do Anexo 02! Erro em %s/%d: %s", uf, ano, e$message))
      return(NULL)
    }
  )

  linha_vazia <- tibble(
    uf = uf, ano = ano, dcl = NA_real_,
    pop = NA_real_, pib_prec_corr = NA_real_, rcl = NA_real_,
    rcla = NA_real_,dtp = NA_real_, dtp_perc_rcla = NA_real_
  )

  if (is.null(anexo1) || nrow(anexo1) == 0) return(linha_vazia)

  if (is.null(anexo2) || nrow(anexo2) == 0) return(linha_vazia)

  # ── Extração: filtra só por cod_conta + coluna ─────────────────────

  dcl <- anexo2 %>%
        filter(
          cod_conta == "DividaConsolidadaLiquida",
          coluna    == "Até o 3º Quadrimestre"
        ) %>% pull(valor) %>% as.numeric() %>% first()

  pop <- anexo1 %>%
    slice(1) %>%
    pull(populacao) %>%
    as.numeric()

  # PIB a preços correntes (Mil Reais) — tabela 5938, variável 37
  pib <- tryCatch(
    get_sidra(
    x          = 5938,
    variable   = 37,
    period     = c(as.character(ano)),
    geo        = "State",
    geo.filter = list("State" = 12),
    format     = 3),
    error = function(e) NULL
  )

  pib_prec_corr <- if (is.null(pib) || nrow(pib) == 0 || !("Valor" %in% names(pib))) { 0 } else {
    as.numeric(pib$Valor) * 1000
  }


  rcl <- anexo1 %>%
    filter(
      cod_conta == "ReceitaCorrenteLiquidaLimiteLegal",
      coluna    == "Valor"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  rcla <- anexo1 %>%
    filter(
      cod_conta == "ReceitaCorrenteLiquidaAjustada",
      coluna    == "Valor"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  dtp <- anexo1 %>%
    filter(
      cod_conta == "DespesaComPessoalTotal",
      coluna    == "Valor"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  dtp_perc_rcla <- anexo1 %>%
    filter(
      cod_conta == "DespesaComPessoalTotal",
      coluna    == "% sobre a RCL Ajustada"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  tibble(
    uf            = uf,
    ano           = ano,
    dcl           = dcl,
    pop           = pop,
    pib_prec_corr = pib_prec_corr,
    rcl           = rcl,
    rcla          = rcla,
    dtp           = dtp,
    dtp_perc_rcla = dtp_perc_rcla
  )
}

# ── Criar a Grade (UF por cada ano i, i+1...) ──────────────────────────────────────

grade <- crossing(estados, ano = anos)   # 162 combinações = 27 UF x (2019 até 2024)

# ── Iteração ──────────────────────────────────────────────────────────────────

df_rgf <- pmap_dfr( list(uf = grade$uf, cod_ibge = grade$cod_ibge, ano = grade$ano),extrair_rgf)


# ── Ordenação ─────────────────────────────────────────────────────────────────

df_rgf <-  df_rgf %>% arrange(uf, ano)

# ── Verificação ───────────────────────────────────────────────────────────────

cat("NA em Colunas:\n")
print(colSums(is.na(df_rgf)))
cat("\n\nLinhas:", nrow(df_rgf), "\n")          # Esperado: 162
print(names(df_rgf))

# ── Exportar ──────────────────────────────────────────────────────────────────
write.csv(df_rgf, "fiscal_2019_2024.csv", row.names = FALSE)

>>> AC | 2019

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2020

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2021

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2022

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2023

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2024

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AL | 2019

Joining with `by = join_by(esfe

NA em Colunas:
           uf           ano           dcl           pop pib_prec_corr 
            0             0             0             0             0 
          rcl          rcla           dtp dtp_perc_rcla 
            0             0             0             0 


Linhas: 162 
[1] "uf"            "ano"           "dcl"           "pop"          
[5] "pib_prec_corr" "rcl"           "rcla"          "dtp"          
[9] "dtp_perc_rcla"
